<a href="https://colab.research.google.com/github/Muhozgu/chlorobiota/blob/main/chlorobiota.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install transformers==4.51.3 accelerate==1.6.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 117.5 MB/s eta 0:00:00


In [24]:
!python /content/chlorobiota.py --images /content/pic.png

  Qwen2.5-VL Plant Identification
Model          : Qwen/Qwen2.5-VL-7B-Instruct
Quantization   : 4-bit NF4
Flash Attn 2   : auto-detect
Images to test : 1
[BEFORE model load] GPU 0 (Tesla T4): allocated=0.00 GB  reserved=0.00 GB  total=14.56 GB
2026-04-29 12:49:01.113783: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

Loading processor from 'Qwen/Qwen2.5-VL-7B-Instruct' …
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Flash Attention 2 not available — falling back to eager 

In [26]:
!pip install transformers==4.51.3 -q

In [27]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL, use_fast=False)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
print("Ready!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Ready!


In [28]:
from PIL import Image

image = Image.open("/content/pic.png").convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": """You are a botanist. Identify this plant and respond in exactly this format:

Common name:
Family name:
Scientific name:
Description: """},
        ],
    },
]

text_prompt = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

inputs = processor(
    text=[text_prompt],
    images=[image],
    return_tensors="pt",
).to(model.device)

print(f"Tokens: {inputs['input_ids'].shape[1]}")

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        repetition_penalty=1.05,
    )

trimmed = generated_ids[0][inputs['input_ids'].shape[1]:]
result = processor.decode(trimmed, skip_special_tokens=True)
print(result)

Tokens: 495


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Common name: Aloe vera  
Family name: Asphodelaceae  
Scientific name: Aloe vera  
Description: Aloe vera is a succulent plant known for its thick, fleshy leaves that are covered with sharp spines along the edges. It is often grown as an ornamental plant or used for its medicinal properties. The plant can be found in various shades of green and is typically grown in a pot indoors or outdoors in warm climates.
